In [11]:
import pandas as pd
import xgboost as xgb
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer

In [12]:
df=pd.read_csv("LaLiga_df.csv")

In [13]:
df.columns

Index(['Unnamed: 0', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HTHG',
       'HTAG', 'HTR', 'Home_GoalsScored_Last5', 'Home_GoalsConceded_Last5',
       'Away_GoalsScored_Last5', 'Away_GoalsConceded_Last5'],
      dtype='object')

In [14]:
cols_to_drop = ['Unnamed: 0','FTHG', 'FTAG', 'FTR', 'HTHG','HTAG', 'HTR']

In [15]:
X=df.drop(columns=cols_to_drop,errors="ignore")

In [16]:
df

,Unnamed: 0,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Home_GoalsScored_Last5,Home_GoalsConceded_Last5,Away_GoalsScored_Last5,Away_GoalsConceded_Last5
0,11,Real Madrid,Ath Bilbao,1,2,A,0.0,0.0,D,5.0,1.0,4.0,0.0
1,12,Betis,Zaragoza,3,1,H,2.0,0.0,H,1.0,1.0,1.0,0.0
2,13,Albacete,Sevilla,3,2,H,2.0,1.0,H,0.0,3.0,0.0,1.0
3,14,Barcelona,Merida,2,2,D,1.0,1.0,D,2.0,0.0,1.0,1.0
4,15,Compostela,La Coruna,4,0,H,2.0,0.0,H,1.0,0.0,3.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
11621,11657,Osasuna,Celta,2,3,A,2.0,1.0,H,0.8,1.4,1.0,1.2
11622,11658,Vallecano,Alaves,1,0,H,0.0,0.0,D,1.4,1.0,1.0,1.0
11623,11659,Mallorca,Levante,1,1,D,0.0,1.0,A,1.2,1.0,1.6,1.6
11624,11660,Real Madrid,Barcelona,2,1,H,2.0,1.0,H,2.4,1.4,2.2,1.4


In [18]:
y_home = df['FTHG']
y_away = df['FTAG']

In [19]:
n = len(X)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train = X.iloc[:train_end]
X_val = X.iloc[train_end:val_end]
X_test = X.iloc[val_end:]

y_home_train = y_home.iloc[:train_end]
y_away_train = y_away.iloc[:train_end]
y_home_val = y_home.iloc[train_end:val_end]
y_away_val = y_away.iloc[train_end:val_end]
y_home_test = y_home.iloc[val_end:]
y_away_test = y_away.iloc[val_end:]

In [20]:
categorical_cols = ['HomeTeam', 'AwayTeam']
numerical_cols = ['Home_GoalsScored_Last5', 'Home_GoalsConceded_Last5', 
                  'Away_GoalsScored_Last5', 'Away_GoalsConceded_Last5']

In [21]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), categorical_cols),
        ('num', SimpleImputer(strategy='mean'), numerical_cols)
    ]
)

In [22]:
pipe_home = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb.XGBRegressor(objective='count:poisson', n_estimators=1500, max_depth=12,random_state=42))
])

pipe_away = Pipeline([
    ('preprocessor', preprocessor),
    ('model', xgb.XGBRegressor(objective='count:poisson', n_estimators=1500, max_depth=12,random_state=42))
])

In [23]:
print("Training Home Model...")
pipe_home.fit(X_train, y_home_train)

print("Training Away Model...")
pipe_away.fit(X_train, y_away_train)

Training Home Model...
Training Away Model...


,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [25]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

def evaluate_model(pipe, X_data, y_true, set_name="Validation"):
    # 1. Generate Predictions
    preds = pipe.predict(X_data)
    
    # 2. Calculate Error Metrics
    mae = mean_absolute_error(y_true, preds)
    
    # FIX: Calculate MSE first, then take square root manually
    mse = mean_squared_error(y_true, preds)
    rmse = np.sqrt(mse) 
    
    print(f"--- {set_name} Set Results ---")
    print(f"   MAE:  {mae:.4f} (Off by about {mae:.2f} goals)")
    print(f"   RMSE: {rmse:.4f}")
    return mae
print("EVALUATING ON VALIDATION SET...")
val_mae_home = evaluate_model(pipe_home, X_val, y_home_val, "Home Goals (Val)")
val_mae_away = evaluate_model(pipe_away, X_val, y_away_val, "Away Goals (Val)")

print("\n" + "="*30 + "\n")

print("EVALUATING ON TEST SET...")
test_mae_home = evaluate_model(pipe_home, X_test, y_home_test, "Home Goals (Test)")
test_mae_away = evaluate_model(pipe_away, X_test, y_away_test, "Away Goals (Test)")

EVALUATING ON VALIDATION SET...
--- Home Goals (Val) Set Results ---
   MAE:  1.1678 (Off by about 1.17 goals)
   RMSE: 1.5179
--- Away Goals (Val) Set Results ---
   MAE:  0.9997 (Off by about 1.00 goals)
   RMSE: 1.3558


EVALUATING ON TEST SET...
--- Home Goals (Test) Set Results ---
   MAE:  1.1194 (Off by about 1.12 goals)
   RMSE: 1.4543
--- Away Goals (Test) Set Results ---
   MAE:  0.9746 (Off by about 0.97 goals)
   RMSE: 1.3142


In [26]:
import joblib

joblib.dump(pipe_home, 'Laliga_model_home_goals.pkl')
joblib.dump(pipe_away, 'Laliga_model_away_goals.pkl')

print("Models saved successfully!")

Models saved successfully!


In [27]:
from sklearn.model_selection import RandomizedSearchCV

# 1. Define the Parameter Grid
# Note: Since we are using a Pipeline, we must add "model__" before the parameter name
param_grid = {
    'model__n_estimators': [100, 300, 500, 700],
    'model__max_depth': [3, 4, 5, 6],
    'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'model__subsample': [0.7, 0.8, 0.9, 1.0],
    'model__colsample_bytree': [0.7, 0.8, 0.9, 1.0]
}

# 2. Setup the Random Search (Optimization Engine)
# scoring='neg_mean_absolute_error' because we want to Minimize MAE
random_search_home = RandomizedSearchCV(
    pipe_home, 
    param_distributions=param_grid, 
    n_iter=20,           # Try 20 random combinations
    cv=3,                # 3-Fold Cross Validation
    scoring='neg_mean_absolute_error',
    verbose=1,
    random_state=42,
    n_jobs=-1            # Use all CPU cores
)

random_search_away = RandomizedSearchCV(
    pipe_away, 
    param_distributions=param_grid, 
    n_iter=20,
    cv=3,
    scoring='neg_mean_absolute_error',
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# 3. Run the Search (This is the heavy lifting)
print("Tuning Home Model (this may take a moment)...")
random_search_home.fit(X_train, y_home_train)
print(f"Best Home Parameters: {random_search_home.best_params_}")
print(f"Best Home MAE: {-random_search_home.best_score_:.4f}")

print("\n Tuning Away Model...")
random_search_away.fit(X_train, y_away_train)
print(f"Best Away Parameters: {random_search_away.best_params_}")
print(f"   Best Away MAE: {-random_search_away.best_score_:.4f}")

Tuning Home Model (this may take a moment)...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best Home Parameters: {'model__subsample': 0.7, 'model__n_estimators': 700, 'model__max_depth': 3, 'model__learning_rate': 0.05, 'model__colsample_bytree': 0.7}
Best Home MAE: 1.0054

 Tuning Away Model...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best Away Parameters: {'model__subsample': 0.7, 'model__n_estimators': 500, 'model__max_depth': 4, 'model__learning_rate': 0.01, 'model__colsample_bytree': 0.8}
   Best Away MAE: 0.8300


In [28]:
# Extract the best models found
best_home_model = random_search_home.best_estimator_
best_away_model = random_search_away.best_estimator_

print("\n🔮 RE-EVALUATING OPTIMIZED MODELS ON TEST SET...")

# Re-run your evaluation function
test_mae_home_opt = evaluate_model(best_home_model, X_test, y_home_test, "Home Goals (Optimized)")
test_mae_away_opt = evaluate_model(best_away_model, X_test, y_away_test, "Away Goals (Optimized)")

# Compare with the old result
print(f"\nImprovement Home: {test_mae_home - test_mae_home_opt:.4f}")
print(f"Improvement Away: {test_mae_away - test_mae_away_opt:.4f}")


🔮 RE-EVALUATING OPTIMIZED MODELS ON TEST SET...
--- Home Goals (Optimized) Set Results ---
   MAE:  0.9674 (Off by about 0.97 goals)
   RMSE: 1.2099
--- Away Goals (Optimized) Set Results ---
   MAE:  0.7951 (Off by about 0.80 goals)
   RMSE: 1.0421

Improvement Home: 0.1520
Improvement Away: 0.1795


In [29]:
joblib.dump(best_home_model, 'Laliga_optimized_model_home.pkl')
joblib.dump(best_away_model, 'Laliga_optimized_model_away.pkl')

print("Optimized models saved successfully!")

Optimized models saved successfully!
